##**Mastering the Pinecone vector database.**

##**GOAL: save text sentence embeddings and perform semantic search in Pinecone using Python code.**

Create a Pinecone Account: Head to 'Pinecone.io' and sign up for a free account. After signing up, you’ll be given an API key.

 To start working with Pinecone,
install 'pinecone' library

In [1]:
%pip install pinecone


Note: you may need to restart the kernel to use updated packages.


From 'pinecone' library import Pinecone, ServerlessSpec.

In [6]:
from pinecone import Pinecone, ServerlessSpec
from dotenv import load_dotenv
import os

load_dotenv()
api_key = os.getenv("PINECONE_API_KEY")
if not api_key:
    raise ValueError("Set the PINECONE_API_KEY environment variable before running this notebook.")

pc = Pinecone(api_key=api_key)
pc


Pinecone(api_key='...XQmo', host='https://api.pinecone.io')

On the Pinecone main page, navigate to the "Get started" section via the left menu. Next, follow the "Quick Start for Developers" guide.
* Initialaze a client.
* Create an index.
* Upsert data.
* Perform a search.









When you perform "Upsert data", add a list of your own text strings.

When you perform "Search", use of your own text string.

In [7]:
index_name = "semantic-search-demo"
if not pc.has_index(index_name):
    pc.create_index(
        name=index_name,
        dimension=384,
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1")
    )

index = pc.Index(index_name)
index


Index(host='https://semantic-search-demo-gq739ur.svc.aped-4627-b74a.pinecone.io')

Print search results.

In [ ]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("paraphrase-MiniLM-L6-v2")
documents = [
    "The product team released a new version to improve reliability and performance.",
    "Customers value fast support, transparent communication, and clear product information.",
    "Our engineering team focuses on building scalable systems and reducing technical debt.",
    "The marketing team uses customer research to refine campaigns and reach the right audience."
]
vectors = [
    (
        str(i),
        model.encode(text).tolist(),
        {"text": text}
    )
    for i, text in enumerate(documents)
]
index.upsert(vectors=vectors)
query = "We are looking for a system that makes customer support faster and more helpful."

query_vector = model.encode(query).tolist()
results = index.query(
    vector=query_vector,
    top_k=3,
    include_metadata=True
)
for match in results["matches"]:
    print("Score:", match["score"])
    print("Document:", match["metadata"]["text"])
    print("-" * 50)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 769.72it/s]


Score: 0.609325707
Document: Customers value fast support, transparent communication, and clear product information.
--------------------------------------------------
Score: 0.488484383
Document: The marketing team uses customer research to refine campaigns and reach the right audience.
--------------------------------------------------
Score: 0.394129127
Document: The product team released a new version to improve reliability and performance.
--------------------------------------------------
